# VectorID (VectorCam edition) — ONNX export for in-browser inference

Converts the EfficientViT-B0 classifier to ONNX so it runs in the phone browser via
onnxruntime-web. One-time step. Output: `vectorcam.onnx` (~8.5MB) -> put in the web app.

Choose which weights: baseline (insectary only, honest 0.65) or augmented (0.955).
GPU not required.

## 0. Setup

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, torch, numpy as np
DRIVE='/content/drive/MyDrive/VectorCam'
!pip -q install timm onnx onnxruntime
device='cpu'  # export on CPU for portability
print('ready')

## 1. Pick the checkpoint
Set CKPT to whichever model you want the app to run.

In [ ]:
import timm
# OPTION A (honest, deploys-today): insectary only
CKPT = os.path.join(DRIVE, 'efficientvit_b0_vectorcam.pt')
# OPTION B (shows the fix, optimistic): uncomment to use augmented
# CKPT = os.path.join(DRIVE, 'efficientvit_b0_augmented_kenya.pt')

ck=torch.load(CKPT, map_location=device)
CLASSES=ck['classes']; MODEL=ck['model']; IMG=ck.get('img',256)
net=timm.create_model(MODEL, pretrained=False, num_classes=len(CLASSES))
net.load_state_dict(ck['state_dict']); net.to(device).eval()
print('loaded', MODEL, '| classes', CLASSES, '| img', IMG)

## 2. Export to ONNX
Fixed 256x256 input. opset 17 is well-supported by onnxruntime-web.

In [ ]:
dummy = torch.randn(1, 3, IMG, IMG, device=device)
onnx_path = '/content/vectorcam.onnx'
torch.onnx.export(
    net, dummy, onnx_path,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17, do_constant_folding=True,
)
print('exported', onnx_path, '| size MB:', round(os.path.getsize(onnx_path)/1e6, 2))

## 3. Verify ONNX matches PyTorch
Run the SAME input through both; outputs must match closely. If they do, the browser
will behave like training. This is the check that catches EfficientViT op issues early.

In [ ]:
import onnxruntime as ort
# pytorch output
with torch.no_grad(): torch_out = net(dummy).cpu().numpy()
# onnx output
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
onnx_out = sess.run(None, {'input': dummy.cpu().numpy()})[0]
diff = np.abs(torch_out - onnx_out).max()
print('max diff pytorch vs onnx:', diff)
print('MATCH ✓' if diff < 1e-3 else 'MISMATCH — investigate (may be an unsupported op)')

## 4. (Recommended) test on a REAL image end-to-end
Confirms the full preprocess->onnx path gives sensible predictions. Uses a Kenya image.

In [ ]:
import zipfile, shutil, pandas as pd
from PIL import Image
if not os.path.exists('/content/data_seg'):
    shutil.copy(os.path.join(DRIVE,'data_seg.zip'),'/content/data_seg.zip')
    with zipfile.ZipFile('/content/data_seg.zip') as z: z.extractall('/content/data_seg')
m=pd.read_csv(os.path.join(DRIVE,'master_seg.csv'))
def rp(r):
    return f"/content/data_seg/image_{int(r['ImageID'])}.jpg"
m['path']=m.apply(rp,axis=1); m=m[m['path'].map(os.path.exists)]
row=m[m['source']=='field'].sample(1, random_state=0).iloc[0]

# EXACT preprocessing the browser must replicate:
mean=np.array([0.485,0.456,0.406]); std=np.array([0.229,0.224,0.225])
img=Image.open(row['path']).convert('RGB')
rs=int(IMG*1.14); img=img.resize((rs,rs))
left=(rs-IMG)//2; img=img.crop((left,left,left+IMG,left+IMG))  # center crop
arr=np.array(img).astype(np.float32)/255.0
arr=(arr-mean)/std
arr=arr.transpose(2,0,1)[None].astype(np.float32)  # NCHW
logits=sess.run(None, {'input': arr})[0][0]
probs=np.exp(logits)/np.exp(logits).sum()
print('true:', row['species'])
for c,p in zip(CLASSES, probs): print(f'  {c}: {p:.3f}')

## 5. Save the ONNX + a labels file to Drive

In [ ]:
shutil.copy(onnx_path, os.path.join(DRIVE,'vectorcam.onnx'))
import json
with open('/content/labels.json','w') as f: json.dump(CLASSES, f)
shutil.copy('/content/labels.json', os.path.join(DRIVE,'labels.json'))
print('saved vectorcam.onnx + labels.json to Drive')
print('\nEXACT preprocessing for the browser (must match):')
print(f'  resize to {rs}x{rs}, center-crop to {IMG}x{IMG}')
print(f'  scale /255, then (x-mean)/std with')
print(f'  mean={mean.tolist()}  std={std.tolist()}')
print(f'  channel order RGB, layout NCHW')

## If step 3 shows MISMATCH or export errors
EfficientViT is a transformer-hybrid; rarely an op isn't ONNX-friendly. Fallback: export
a PURE CNN checkpoint instead (MobileNetV3 or convnext) — they convert cleanly and run
great in-browser. Just set CKPT to that model and re-run. Tell me and I'll adjust.